# CatCare AI — treinamento com Oxford-IIIT Pet

O Colab baixará o [Oxford-IIIT Pet](https://robots.ox.ac.uk/~vgg/data/pets/) automaticamente. Você **não precisa enviar fotos nem `labels.csv`**. O download do dataset é de aproximadamente 800 MB e os pesos iniciais do EfficientNet-B0 também serão baixados.

Antes de executar, escolha **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**. Depois rode as células em ordem.

Este treino aprende **12 raças de gatos**. O Oxford não tem uma classe `srd` nem os rótulos de características, cores, padrões e comprimento de pelagem usados pelo CatCare AI. No backend, uma previsão de raça com probabilidade abaixo de `ML_BREED_MIN_CONFIDENCE` (padrão: 0,85) vira `SRD` provisório. Essa regra não detecta todos os SRDs: o classificador pode escolher uma raça com probabilidade alta para um gato sem raça definida. Por isso, o administrador precisa revisar a sugestão. As saídas de pelagem continuarão sem previsão até serem treinadas com outro dataset rotulado.


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

REPO = Path('/content/catcare-ai')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Danilogggs/gatitos.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'backend'))
from app.ml.train_oxford import OXFORD_BREEDS
print('GPU disponível:', torch.cuda.is_available())
print('Raças treinadas:', ', '.join(OXFORD_BREEDS))


## Baixe os dados e treine

A próxima célula baixa o Oxford, separa apenas os gatos, divide `trainval` em treino e validação e usa `test` para uma avaliação final. O checkpoint com melhor acurácia de validação será salvo em `/content/catcare-oxford.pt`. O tempo depende da GPU e do download.


In [ ]:
EPOCHS = 8
BATCH_SIZE = 16
MODEL_PATH = Path('/content/catcare-oxford.pt')
subprocess.run([
    sys.executable, '-m', 'app.ml.train_oxford',
    '--data-dir', '/content/oxford-data', '--output', str(MODEL_PATH),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
], cwd=REPO / 'backend', check=True)
print('Checkpoint criado:', MODEL_PATH)


## Baixe o checkpoint para o seu computador

Execute a célula abaixo. Depois coloque o arquivo em um caminho acessível ao backend, configure `ML_MODE=real` e `ML_MODEL_PATH` no `.env` local. Não envie o checkpoint ou o `.env` ao GitHub.


In [ ]:
from google.colab import files
files.download(str(MODEL_PATH))
